In [1]:
!pip install scipy
# ============================================================
# Q1 ATTENTION BENCHMARK — SCIENTIFIC PROOF VERSION
# ------------------------------------------------------------
# This version keeps your original Q1 design, but adds:
# 1) clean and attacked row-level predictions
# 2) direct attack-drop proof (clean vs attacked)
# 3) flip-rate proof
# 4) McNemar exact paired significance test
# 5) bootstrap confidence intervals for attack drop
# 6) false-positive / false-negative pull analysis
# 7) disagreement-slice and gold-label-slice analysis
# 8) shared harmed-row exports for qualitative defense
# ============================================================

import hashlib
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from tqdm.auto import tqdm

import kaggle_benchmarks as kbench

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600
MAX_CHARS = 6000
BOOT_N = 5000

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    "qwen/qwen3-235b-a22b-instruct-2507"
]

OUT_DIR = Path("/kaggle/working/epu_attention_q1_scientific_outputs_fixed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MISSING_SENTINEL = -1

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "mention_foreign",
    "mainly_foreign",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]

for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)

clean = df.loc[mask].copy()

clean["article_key"] = clean["article_key"].astype(str)
missing_key_mask = clean["article_key"].isin(["nan", "None", ""])
if missing_key_mask.any():
    clean.loc[missing_key_mask, "article_key"] = [
        f"generated_key_{i}" for i in range(missing_key_mask.sum())
    ]

clean["gold_epu"] = clean["gold_epu"].astype(int)
clean["gold_mention_foreign"] = (clean["mention_foreign"].fillna(0) >= 0.5).astype(int)
clean["gold_mainly_foreign"] = (clean["mainly_foreign"].fillna(0) >= 0.5).astype(int)
clean["gold_who"] = clean["gold_who"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_actions"] = clean["gold_actions"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_effects"] = clean["gold_effects"].fillna(MISSING_SENTINEL).astype(int)
clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)

clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print(clean["gold_epu"].value_counts(dropna=False).sort_index())
print(clean["disagreement_flag"].value_counts(dropna=False).sort_index())

# ---------------------------
# BALANCED SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)

    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)

    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed + 1)

    return (
        pd.concat([pos_s, neg_s], axis=0)
        .sample(frac=1, random_state=seed + 2)
        .reset_index(drop=True)
    )

pilot = balanced_sample(clean, N_ROWS, SEED).copy()

print("\nPilot class balance:")
print(pilot["gold_epu"].value_counts().sort_index())
print("\nPilot disagreement balance:")
print(pilot["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# DISTRACTORS
# ---------------------------
DISTRACTORS = [
    "A late playoff result forced coaches to reshuffle their lines, while reporters focused on injuries, locker-room reactions, and the pressure going into the next match.",
    "Heavy rain and strong winds disrupted travel across several cities, with flooded roads, delayed buses, and repeated service alerts during the morning commute.",
    "Stocks swung through afternoon trading after several companies posted mixed earnings, as investors reacted to forecasts, margins, and shifting market sentiment.",
    "Candidates spent the weekend rallying in key districts, drawing large crowds, responding to criticism, and trying to regain momentum before the next debate.",
    "Diplomats met for emergency talks after border tensions rose overnight, while officials issued conflicting statements about the likely direction of negotiations.",
    "A major streaming release was delayed after contract talks stalled, leading entertainment reporters to speculate about scheduling changes and audience response.",
    "Researchers disagreed over the interpretation of a new medical study, with some calling the findings promising and others urging caution until more evidence is available.",
    "Downtown businesses reported uneven foot traffic during a busy weekend, as event organizers pointed to weather, parking shortages, and road closures as possible reasons.",
]

BAD_DISTRACTOR_TERMS = [
    "epu",
    "economic policy",
    "policy uncertainty",
    "target article",
    "classify",
    "irrelevant",
    "not evidence",
    "public-policy",
    "distractor snippet",
]

for _d in DISTRACTORS:
    d_low = _d.lower()
    assert not any(term in d_low for term in BAD_DISTRACTOR_TERMS), f"Bad distractor leakage found: {_d}"

def stable_index(key: str, mod: int) -> int:
    h = hashlib.md5(str(key).encode("utf-8")).hexdigest()
    return int(h[:8], 16) % mod

def pick_distractor(article_key: str, suffix: str) -> str:
    idx = stable_index(f"{article_key}_{suffix}", len(DISTRACTORS))
    return DISTRACTORS[idx]

def smart_truncate(text: str, max_chars: int = 6000) -> str:
    text = str(text)
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head - 32
    return text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]

pilot["content_for_eval"] = pilot["content"].astype(str).apply(lambda x: smart_truncate(x, MAX_CHARS))
pilot["distractor_pre"] = pilot["article_key"].astype(str).apply(lambda x: pick_distractor(x, "pre"))
pilot["distractor_post"] = pilot["article_key"].astype(str).apply(lambda x: pick_distractor(x, "post"))

pilot_path = OUT_DIR / "pilot_rows_600.csv"
pilot.to_csv(pilot_path, index=False)

# ---------------------------
# HELPERS
# ---------------------------
def maybe_int(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        return None

def binary_match(pred, gold, missing_sentinel=MISSING_SENTINEL):
    gold_i = maybe_int(gold)
    pred_i = maybe_int(pred)

    if gold_i is None or gold_i == missing_sentinel:
        return None
    if pred_i is None:
        return 0.0
    return 1.0 if pred_i == gold_i else 0.0

def build_prompt(article_text: str, distractor_pre: str = "", distractor_post: str = "") -> str:
    surrounding_blocks = []
    if distractor_pre:
        surrounding_blocks.append("CONTEXT BLOCK A\n" + distractor_pre)
    surrounding_blocks.append("TARGET ARTICLE START\n" + article_text + "\nTARGET ARTICLE END")
    if distractor_post:
        surrounding_blocks.append("CONTEXT BLOCK B\n" + distractor_post)

    joined_text = "\n\n".join(surrounding_blocks)

    return (
        "You will read multiple news-style text blocks.\n\n"
        "Task:\n"
        "Classify only the article between TARGET ARTICLE START and TARGET ARTICLE END.\n"
        "Return a judgment for that target article only.\n\n"
        "Labeling rule:\n"
        "- EPU = 1 only if the TARGET ARTICLE discusses uncertainty about government policy, "
        "policy decisions, possible policy actions, political outcomes affecting policy, "
        "or the economic effects of policy.\n"
        "- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events "
        "unless the uncertainty is specifically tied to economic policy.\n\n"
        "Return ONLY valid JSON.\n"
        "No markdown. No explanation. No code fence.\n\n"
        "Return exactly these keys:\n"
        "{\n"
        '  "epu": 0 or 1,\n'
        '  "who": 0 or 1 or null,\n'
        '  "actions": 0 or 1 or null,\n'
        '  "effects": 0 or 1 or null,\n'
        '  "mention_foreign": 0 or 1 or null,\n'
        '  "mainly_foreign": 0 or 1 or null,\n'
        '  "confidence": number between 0 and 1,\n'
        '  "focus_excerpt": "short quote from target article only"\n'
        "}\n\n"
        f"{joined_text}"
    ).strip()

def default_out():
    return {
        "epu": None,
        "who": None,
        "actions": None,
        "effects": None,
        "mention_foreign": None,
        "mainly_foreign": None,
        "confidence": 0.0,
        "focus_excerpt": "",
        "raw_text": "",
    }

def parse_llm_json(raw_text):
    out = default_out()
    out["raw_text"] = "" if raw_text is None else str(raw_text)[:3000]

    if raw_text is None:
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return out
    except Exception:
        return out

    for k in out:
        if k in obj and k != "raw_text":
            out[k] = obj[k]

    for k in ["epu", "who", "actions", "effects", "mention_foreign", "mainly_foreign"]:
        out[k] = maybe_int(out[k])

    try:
        out["confidence"] = float(out["confidence"])
    except Exception:
        out["confidence"] = 0.0

    out["confidence"] = max(0.0, min(1.0, out["confidence"]))
    out["focus_excerpt"] = "" if out["focus_excerpt"] is None else str(out["focus_excerpt"])[:250]
    return out

def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:3000]}"
        return out

def excerpt_in_target(excerpt: str, article_text: str) -> int:
    excerpt = "" if excerpt is None else str(excerpt).strip()
    article_text = "" if article_text is None else str(article_text)
    if len(excerpt) < 12:
        return 0
    return int(excerpt in article_text)

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def score_composite(clean_out: dict, attacked_out: dict, row: pd.Series) -> tuple[float, dict]:
    gold_epu = maybe_int(row["gold_epu"])
    gold_who = maybe_int(row["gold_who"])
    gold_actions = maybe_int(row["gold_actions"])
    gold_effects = maybe_int(row["gold_effects"])
    gold_mention_foreign = maybe_int(row["gold_mention_foreign"])
    gold_mainly_foreign = maybe_int(row["gold_mainly_foreign"])

    score = 0.0
    denom = 0.0
    comp = {}

    m = binary_match(clean_out["epu"], gold_epu)
    comp["clean_epu_correct"] = np.nan if m is None else float(m)
    if m is not None:
        score += 0.40 * m
        denom += 0.40

    m = binary_match(attacked_out["epu"], gold_epu)
    comp["attacked_epu_correct"] = np.nan if m is None else float(m)
    if m is not None:
        score += 0.30 * m
        denom += 0.30

    clean_epu = maybe_int(clean_out["epu"])
    attacked_epu = maybe_int(attacked_out["epu"])
    stability = np.nan
    if clean_epu is not None and attacked_epu is not None:
        stability = 1.0 if clean_epu == attacked_epu else 0.0
        score += 0.10 * stability
        denom += 0.10
    comp["stable_epu_flag"] = stability

    aux_specs = [
        ("who", attacked_out["who"], gold_who),
        ("actions", attacked_out["actions"], gold_actions),
        ("effects", attacked_out["effects"], gold_effects),
        ("mention_foreign", attacked_out["mention_foreign"], gold_mention_foreign),
        ("mainly_foreign", attacked_out["mainly_foreign"], gold_mainly_foreign),
    ]
    aux_weight_each = 0.04

    aux_correct_sum = 0.0
    aux_correct_n = 0.0

    for label_name, pred, gold in aux_specs:
        m = binary_match(pred, gold)
        comp[f"attacked_{label_name}_correct"] = np.nan if m is None else float(m)
        if m is not None:
            aux_correct_sum += float(m)
            aux_correct_n += 1.0
            score += aux_weight_each * m
            denom += aux_weight_each

    comp["attacked_aux_mean"] = np.nan if aux_correct_n == 0 else aux_correct_sum / aux_correct_n
    comp["composite_score"] = 0.0 if denom <= 0 else float(score / denom)
    return comp["composite_score"], comp

# ---------------------------
# EVAL DATA
# ---------------------------
eval_cols = [
    "article_key",
    "content_for_eval",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "gold_mention_foreign",
    "gold_mainly_foreign",
    "distractor_pre",
    "distractor_post",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]

eval_df = pilot[eval_cols].copy()

for col in [
    "gold_who",
    "gold_actions",
    "gold_effects",
    "gold_mention_foreign",
    "gold_mainly_foreign",
]:
    eval_df[col] = eval_df[col].fillna(MISSING_SENTINEL).astype(int)

eval_df["gold_epu"] = eval_df["gold_epu"].astype(int)
eval_df["disagreement_flag"] = eval_df["disagreement_flag"].fillna(0).astype(int)
eval_df["newspaper"] = eval_df["newspaper"].fillna("").astype(str)
eval_df["year"] = pd.to_numeric(eval_df["year"], errors="coerce")
eval_df["content_len"] = pd.to_numeric(eval_df["content_len"], errors="coerce")

core_no_nan_cols = [
    "article_key",
    "content_for_eval",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "gold_mention_foreign",
    "gold_mainly_foreign",
    "distractor_pre",
    "distractor_post",
    "disagreement_flag",
]
assert eval_df[core_no_nan_cols].isna().sum().sum() == 0, "eval_df still contains NaN in required columns"

# ---------------------------
# MANUAL Q1 RUN
# ---------------------------
records = []
model_handles = {name: kbench.llms[name] for name in MODEL_NAMES}

for model_name in MODEL_NAMES:
    llm = model_handles[model_name]
    print(f"\nRunning {model_name} ...")

    for row in tqdm(eval_df.to_dict(orient="records"), total=len(eval_df), desc=model_name):
        clean_prompt = build_prompt(row["content_for_eval"])
        attacked_prompt = build_prompt(
            row["content_for_eval"],
            row["distractor_pre"],
            row["distractor_post"],
        )

        clean_out = model_json_call(llm, clean_prompt)
        attacked_out = model_json_call(llm, attacked_prompt)

        composite_score, comp = score_composite(clean_out, attacked_out, row)

        clean_pred = maybe_int(clean_out["epu"])
        attacked_pred = maybe_int(attacked_out["epu"])
        gold_epu = maybe_int(row["gold_epu"])

        clean_correct = comp["clean_epu_correct"]
        attacked_correct = comp["attacked_epu_correct"]
        stable_flag = comp["stable_epu_flag"]

        attack_hurt = int(clean_correct == 1.0 and attacked_correct == 0.0) if not pd.isna(clean_correct) and not pd.isna(attacked_correct) else 0
        attack_helped = int(clean_correct == 0.0 and attacked_correct == 1.0) if not pd.isna(clean_correct) and not pd.isna(attacked_correct) else 0
        prediction_flip = int((clean_pred is not None) and (attacked_pred is not None) and (clean_pred != attacked_pred))

        clean_fp = int(gold_epu == 0 and clean_pred == 1) if clean_pred is not None else 0
        attacked_fp = int(gold_epu == 0 and attacked_pred == 1) if attacked_pred is not None else 0
        clean_fn = int(gold_epu == 1 and clean_pred == 0) if clean_pred is not None else 0
        attacked_fn = int(gold_epu == 1 and attacked_pred == 0) if attacked_pred is not None else 0

        rec = {
            "llm_name": model_name,
            "article_key": row["article_key"],
            "gold_epu": row["gold_epu"],
            "gold_who": row["gold_who"],
            "gold_actions": row["gold_actions"],
            "gold_effects": row["gold_effects"],
            "gold_mention_foreign": row["gold_mention_foreign"],
            "gold_mainly_foreign": row["gold_mainly_foreign"],
            "disagreement_flag": row["disagreement_flag"],
            "newspaper": row.get("newspaper", ""),
            "year": row.get("year", np.nan),
            "content_len": row.get("content_len", np.nan),
            "content_for_eval": row["content_for_eval"],
            "distractor_pre": row["distractor_pre"],
            "distractor_post": row["distractor_post"],

            "clean_epu_pred": clean_pred,
            "attacked_epu_pred": attacked_pred,
            "clean_epu_correct": clean_correct,
            "attacked_epu_correct": attacked_correct,
            "stable_epu_flag": stable_flag,
            "prediction_flip": prediction_flip,
            "attack_hurt": attack_hurt,
            "attack_helped": attack_helped,

            "clean_fp": clean_fp,
            "attacked_fp": attacked_fp,
            "clean_fn": clean_fn,
            "attacked_fn": attacked_fn,

            "clean_who_pred": maybe_int(clean_out["who"]),
            "clean_actions_pred": maybe_int(clean_out["actions"]),
            "clean_effects_pred": maybe_int(clean_out["effects"]),
            "clean_mention_foreign_pred": maybe_int(clean_out["mention_foreign"]),
            "clean_mainly_foreign_pred": maybe_int(clean_out["mainly_foreign"]),

            "attacked_who_pred": maybe_int(attacked_out["who"]),
            "attacked_actions_pred": maybe_int(attacked_out["actions"]),
            "attacked_effects_pred": maybe_int(attacked_out["effects"]),
            "attacked_mention_foreign_pred": maybe_int(attacked_out["mention_foreign"]),
            "attacked_mainly_foreign_pred": maybe_int(attacked_out["mainly_foreign"]),

            "attacked_who_correct": comp["attacked_who_correct"],
            "attacked_actions_correct": comp["attacked_actions_correct"],
            "attacked_effects_correct": comp["attacked_effects_correct"],
            "attacked_mention_foreign_correct": comp["attacked_mention_foreign_correct"],
            "attacked_mainly_foreign_correct": comp["attacked_mainly_foreign_correct"],
            "attacked_aux_mean": comp["attacked_aux_mean"],

            "clean_confidence": safe_float(clean_out["confidence"]),
            "attacked_confidence": safe_float(attacked_out["confidence"]),
            "clean_focus_excerpt": clean_out["focus_excerpt"],
            "attacked_focus_excerpt": attacked_out["focus_excerpt"],
            "clean_excerpt_in_target": excerpt_in_target(clean_out["focus_excerpt"], row["content_for_eval"]),
            "attacked_excerpt_in_target": excerpt_in_target(attacked_out["focus_excerpt"], row["content_for_eval"]),
            "clean_raw_json": clean_out["raw_text"],
            "attacked_raw_json": attacked_out["raw_text"],

            "composite_score": composite_score,
        }
        records.append(rec)

detailed_df = pd.DataFrame(records)

raw_path = OUT_DIR / "q1_results_detailed_600x4.csv"
detailed_df.to_csv(raw_path, index=False)

# ---------------------------
# STATISTICS
# ---------------------------
def wilson_ci(successes: int, n: int, z: float = 1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = successes / n
    denom = 1.0 + (z**2) / n
    center = (phat + (z**2) / (2 * n)) / denom
    margin = (z / denom) * math.sqrt((phat * (1 - phat) / n) + (z**2) / (4 * n**2))
    return (max(0.0, center - margin), min(1.0, center + margin))

def bootstrap_mean_ci(values, n_boot=5000, seed=42, alpha=0.05):
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sample = rng.choice(arr, size=n, replace=True)
        means[i] = sample.mean()
    return (float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2)))

def paired_bootstrap_diff_ci(clean_values, attacked_values, n_boot=5000, seed=42, alpha=0.05):
    a = np.asarray(clean_values, dtype=float)
    b = np.asarray(attacked_values, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    n = len(a)
    if n == 0:
        return (np.nan, np.nan)
    diffs = a - b
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = diffs[idx].mean()
    return (float(np.quantile(boot, alpha / 2)), float(np.quantile(boot, 1 - alpha / 2)))

def exact_mcnemar(clean_correct, attacked_correct):
    x = np.asarray(clean_correct, dtype=float)
    y = np.asarray(attacked_correct, dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x = x[mask].astype(int)
    y = y[mask].astype(int)
    b = int(np.sum((x == 1) & (y == 0)))  # hurt
    c = int(np.sum((x == 0) & (y == 1)))  # helped
    n = b + c
    if n == 0:
        return {
            "b_hurt": b,
            "c_helped": c,
            "discordant_n": n,
            "mcnemar_p": 1.0,
            "hurt_help_ratio": np.nan,
        }
    pval = binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    ratio = np.inf if c == 0 and b > 0 else (b / c if c > 0 else np.nan)
    return {
        "b_hurt": b,
        "c_helped": c,
        "discordant_n": n,
        "mcnemar_p": float(pval),
        "hurt_help_ratio": ratio,
    }

def classify_distraction(drop_pp, flip_rate, pval):
    if pd.isna(drop_pp) or pd.isna(flip_rate):
        return "undetermined"
    if (drop_pp >= 0.10 and flip_rate >= 0.15 and pval < 0.05):
        return "strong distraction"
    if (drop_pp >= 0.03 and flip_rate >= 0.05 and pval < 0.05):
        return "moderate distraction"
    if drop_pp < 0.03 and flip_rate < 0.05:
        return "little distraction"
    return "mixed / borderline"

def summarize_group(g: pd.DataFrame, group_name: str, group_value):
    out = {
        "group_name": group_name,
        "group_value": group_value,
        "n_rows": len(g),
    }

    # Clean accuracy
    clean_vec = g["clean_epu_correct"].astype(float).to_numpy()
    attacked_vec = g["attacked_epu_correct"].astype(float).to_numpy()

    clean_n = int(np.sum(~np.isnan(clean_vec)))
    attacked_n = int(np.sum(~np.isnan(attacked_vec)))

    clean_acc = float(np.nanmean(clean_vec)) if clean_n > 0 else np.nan
    attacked_acc = float(np.nanmean(attacked_vec)) if attacked_n > 0 else np.nan

    clean_k = int(np.nansum(clean_vec)) if clean_n > 0 else 0
    attacked_k = int(np.nansum(attacked_vec)) if attacked_n > 0 else 0

    clean_lo, clean_hi = wilson_ci(clean_k, clean_n)
    attacked_lo, attacked_hi = wilson_ci(attacked_k, attacked_n)

    out["clean_acc"] = clean_acc
    out["clean_acc_ci_low"] = clean_lo
    out["clean_acc_ci_high"] = clean_hi
    out["attacked_acc"] = attacked_acc
    out["attacked_acc_ci_low"] = attacked_lo
    out["attacked_acc_ci_high"] = attacked_hi

    # Direct attack drop
    attack_drop_pp = clean_acc - attacked_acc if clean_n > 0 and attacked_n > 0 else np.nan
    drop_lo, drop_hi = paired_bootstrap_diff_ci(clean_vec, attacked_vec, n_boot=BOOT_N, seed=SEED)
    out["attack_drop_pp"] = attack_drop_pp
    out["attack_drop_ci_low"] = drop_lo
    out["attack_drop_ci_high"] = drop_hi

    # Flip proof
    flip_n = int(g["prediction_flip"].notna().sum())
    flip_k = int(g["prediction_flip"].fillna(0).sum())
    flip_rate = flip_k / flip_n if flip_n > 0 else np.nan
    flip_lo, flip_hi = wilson_ci(flip_k, flip_n)

    out["flip_rate"] = flip_rate
    out["flip_rate_ci_low"] = flip_lo
    out["flip_rate_ci_high"] = flip_hi

    # Hurt/help proof
    hurt_k = int(g["attack_hurt"].sum())
    help_k = int(g["attack_helped"].sum())
    hurt_rate = hurt_k / len(g) if len(g) > 0 else np.nan
    help_rate = help_k / len(g) if len(g) > 0 else np.nan

    hurt_lo, hurt_hi = wilson_ci(hurt_k, len(g))
    help_lo, help_hi = wilson_ci(help_k, len(g))

    out["attack_hurt_rate"] = hurt_rate
    out["attack_hurt_ci_low"] = hurt_lo
    out["attack_hurt_ci_high"] = hurt_hi
    out["attack_helped_rate"] = help_rate
    out["attack_helped_ci_low"] = help_lo
    out["attack_helped_ci_high"] = help_hi

    # Paired significance
    mc = exact_mcnemar(clean_vec, attacked_vec)
    out.update(mc)

    # Error inflation
    if pd.notna(clean_acc) and pd.notna(attacked_acc) and clean_acc < 1.0:
        clean_err = 1.0 - clean_acc
        attacked_err = 1.0 - attacked_acc
        out["relative_error_increase"] = (attacked_err / clean_err - 1.0) if clean_err > 0 else np.nan
    else:
        out["relative_error_increase"] = np.nan

    # Class-specific pull
    gold0 = g[g["gold_epu"] == 0].copy()
    gold1 = g[g["gold_epu"] == 1].copy()

    if len(gold0) > 0:
        clean_fp_rate = gold0["clean_fp"].mean()
        attacked_fp_rate = gold0["attacked_fp"].mean()
        out["clean_fp_rate_gold0"] = clean_fp_rate
        out["attacked_fp_rate_gold0"] = attacked_fp_rate
        out["fp_pull_pp"] = attacked_fp_rate - clean_fp_rate
    else:
        out["clean_fp_rate_gold0"] = np.nan
        out["attacked_fp_rate_gold0"] = np.nan
        out["fp_pull_pp"] = np.nan

    if len(gold1) > 0:
        clean_fn_rate = gold1["clean_fn"].mean()
        attacked_fn_rate = gold1["attacked_fn"].mean()
        out["clean_fn_rate_gold1"] = clean_fn_rate
        out["attacked_fn_rate_gold1"] = attacked_fn_rate
        out["fn_pull_pp"] = attacked_fn_rate - clean_fn_rate
    else:
        out["clean_fn_rate_gold1"] = np.nan
        out["attacked_fn_rate_gold1"] = np.nan
        out["fn_pull_pp"] = np.nan

    # Other support
    out["stable_rate"] = g["stable_epu_flag"].mean()
    out["composite_score_mean"] = g["composite_score"].mean()
    out["attacked_aux_mean"] = g["attacked_aux_mean"].mean()
    out["clean_confidence_mean"] = g["clean_confidence"].mean()
    out["attacked_confidence_mean"] = g["attacked_confidence"].mean()
    out["clean_excerpt_in_target_rate"] = g["clean_excerpt_in_target"].mean()
    out["attacked_excerpt_in_target_rate"] = g["attacked_excerpt_in_target"].mean()

    out["distraction_level"] = classify_distraction(
        out["attack_drop_pp"],
        out["flip_rate"],
        out["mcnemar_p"],
    )
    return out

# ---------------------------
# SUMMARY TABLES
# ---------------------------
summary_overall = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "overall", "all"))
        for model_name, g in detailed_df.groupby("llm_name", sort=False)
    ]
).sort_values(["attack_drop_pp", "flip_rate"], ascending=[False, False])

summary_overall_path = OUT_DIR / "summary_overall_scientific_600x4.csv"
summary_overall.to_csv(summary_overall_path, index=False)

summary_by_disagreement = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "disagreement_flag", int(flag)))
        for (model_name, flag), g in detailed_df.groupby(["llm_name", "disagreement_flag"], sort=False)
    ]
)

summary_by_disagreement_path = OUT_DIR / "summary_by_disagreement_scientific_600x4.csv"
summary_by_disagreement.to_csv(summary_by_disagreement_path, index=False)

summary_by_gold_epu = pd.DataFrame(
    [
        dict({"llm_name": model_name}, **summarize_group(g, "gold_epu", int(flag)))
        for (model_name, flag), g in detailed_df.groupby(["llm_name", "gold_epu"], sort=False)
    ]
)

summary_by_gold_epu_path = OUT_DIR / "summary_by_gold_epu_scientific_600x4.csv"
summary_by_gold_epu.to_csv(summary_by_gold_epu_path, index=False)

# Transition tables
transition_rows = []
for (model_name, gold_epu), g in detailed_df.groupby(["llm_name", "gold_epu"], sort=False):
    tmp = (
        g.assign(
            clean_label=g["clean_epu_pred"].fillna(-9).astype(int),
            attacked_label=g["attacked_epu_pred"].fillna(-9).astype(int),
        )
        .groupby(["clean_label", "attacked_label"])
        .size()
        .reset_index(name="count")
    )
    tmp["llm_name"] = model_name
    tmp["gold_epu"] = gold_epu
    transition_rows.append(tmp)

transition_df = pd.concat(transition_rows, ignore_index=True)
transition_path = OUT_DIR / "transition_counts_clean_to_attacked_600x4.csv"
transition_df.to_csv(transition_path, index=False)

# Most harmed rows
harmed_rows = (
    detailed_df.groupby(["article_key", "gold_epu", "disagreement_flag"], as_index=False)
    .agg(
        harmed_models=("attack_hurt", "sum"),
        helped_models=("attack_helped", "sum"),
        mean_composite=("composite_score", "mean"),
        mean_clean_correct=("clean_epu_correct", "mean"),
        mean_attacked_correct=("attacked_epu_correct", "mean"),
        content_for_eval=("content_for_eval", "first"),
        distractor_pre=("distractor_pre", "first"),
        distractor_post=("distractor_post", "first"),
    )
    .sort_values(["harmed_models", "mean_attacked_correct"], ascending=[False, True])
)
harmed_rows_path = OUT_DIR / "most_harmed_rows_shared_600x4.csv"
harmed_rows.to_csv(harmed_rows_path, index=False)

# Model-specific worst distractor failures: clean correct -> attacked wrong
worst_attack_cases = (
    detailed_df[detailed_df["attack_hurt"] == 1]
    .sort_values(["llm_name", "attacked_confidence", "composite_score"], ascending=[True, False, True])
)
worst_attack_cases_path = OUT_DIR / "worst_attack_failures_by_model_600x4.csv"
worst_attack_cases.to_csv(worst_attack_cases_path, index=False)

# Manifest
manifest = pd.DataFrame({
    "file_name": [
        raw_path.name,
        summary_overall_path.name,
        summary_by_disagreement_path.name,
        summary_by_gold_epu_path.name,
        transition_path.name,
        harmed_rows_path.name,
        worst_attack_cases_path.name,
        pilot_path.name,
    ],
    "description": [
        "Row-level clean and attacked predictions, correctness flags, confidences, excerpts, and composite score.",
        "Main scientific proof table: clean accuracy, attacked accuracy, attack drop, flip rate, McNemar p-value, FP/FN pull, and distraction label.",
        "Same scientific proof metrics split by disagreement_flag.",
        "Same scientific proof metrics split by gold_epu.",
        "Transition counts from clean prediction to attacked prediction, by model and gold_epu.",
        "Rows harmed by distractors across multiple models, for qualitative evidence.",
        "Model-specific rows where distractors changed a clean-correct prediction into an attacked-wrong prediction.",
        "The exact 600-row pilot used for the run.",
    ]
})
manifest_path = OUT_DIR / "manifest_scientific_600x4.csv"
manifest.to_csv(manifest_path, index=False)

print("\nSaved files:")
for p in [
    raw_path,
    summary_overall_path,
    summary_by_disagreement_path,
    summary_by_gold_epu_path,
    transition_path,
    harmed_rows_path,
    worst_attack_cases_path,
    pilot_path,
    manifest_path,
]:
    print(p)

print("\nOverall proof table:")
display(summary_overall)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/35.3 MB ? eta -:--:--

   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/35.3 MB 40.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 35.1/35.3 MB 103.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 73.0 MB/s  0:00:00


Loaded rows: 800
Clean usable rows: 800
gold_epu
0    400
1    400
Name: count, dtype: int64
disagreement_flag
0    585
1    215
Name: count, dtype: int64

Pilot class balance:
gold_epu
0    300
1    300
Name: count, dtype: int64

Pilot disagreement balance:
disagreement_flag
0    429
1    171
Name: count, dtype: int64



Running qwen/qwen3-235b-a22b-instruct-2507 ...


qwen/qwen3-235b-a22b-instruct-2507:   0%|          | 0/600 [00:00<?, ?it/s]


Saved files:
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/q1_results_detailed_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/summary_overall_scientific_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/summary_by_disagreement_scientific_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/summary_by_gold_epu_scientific_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/transition_counts_clean_to_attacked_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/most_harmed_rows_shared_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/worst_attack_failures_by_model_600x4.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/pilot_rows_600.csv
/kaggle/working/epu_attention_q1_scientific_outputs_fixed/manifest_scientific_600x4.csv

Overall proof table:


,llm_name,group_name,group_value,n_rows,clean_acc,clean_acc_ci_low,clean_acc_ci_high,attacked_acc,attacked_acc_ci_low,attacked_acc_ci_high,...,attacked_fn_rate_gold1,fn_pull_pp,stable_rate,composite_score_mean,attacked_aux_mean,clean_confidence_mean,attacked_confidence_mean,clean_excerpt_in_target_rate,attacked_excerpt_in_target_rate,distraction_level
0,qwen/qwen3-235b-a22b-instruct-2507,overall,all,600,0.665,0.62629,0.701611,0.658333,0.619485,0.695167,...,0.46,-0.036667,0.924242,0.653478,0.491333,0.934917,0.937167,0.588333,0.638333,mixed / borderline
